In [ ]:
# !pip install tensorflow==2.15
!pip install tensorflow
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential, Model
from keras.utils import Sequence
from keras.optimizers import Adam
from tensorflow.keras.utils import plot_model
from keras.callbacks import ModelCheckpoint, EarlyStopping
from keras.layers import Input, Conv1D, Conv1DTranspose,AveragePooling1D, MaxPooling1D,UpSampling1D,LeakyReLU, ReLU, concatenate, Dropout,BatchNormalization,Activation
from pathlib import Path
from IPython.display import Image
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy
import h5py
import random
from tensorflow.keras.losses import MeanAbsoluteError
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from scipy.stats import pearsonr
from keras.callbacks import ReduceLROnPlateau
from google.colab import drive
import os
drive.mount('/content/drive',force_remount=True)

In [ ]:
import tensorflow as tf
tf.__version__

'2.19.0'

In [ ]:
Train_Address = "/content/drive/MyDrive/Spike_Restoration/Main_Work_not_Proposal/Splitted_Data_for_paper1/SNDR/Train.mat"
Info = scipy.io.whosmat(Train_Address)
print(Info)
datam = scipy.io.loadmat(Train_Address)
Train_Data_Y_1 = np.asarray(datam ['Data'])

Val_Address = "/content/drive/MyDrive/Spike_Restoration/Main_Work_not_Proposal/Splitted_Data_for_paper1/SNDR/Val.mat"
Info = scipy.io.whosmat(Val_Address)
print(Info)
datam = scipy.io.loadmat(Val_Address)
Val_Data_Y = np.asarray(datam ['Data'])

VD sampling pattern - generated in MATLAB

In [ ]:
Mask_32_64_Address = '/content/drive/MyDrive/Spike_Restoration/Main_Work_not_Proposal/Sampling_Pattern_Design/Mask_32.mat'
Info = scipy.io.whosmat(Mask_32_64_Address)
print(Info)
datam = scipy.io.loadmat(Mask_32_64_Address)
Mask_32_64 = np.asarray(datam ['pattern'])
print(Mask_32_64)

Mask_16_64_Address = '/content/drive/MyDrive/Spike_Restoration/Main_Work_not_Proposal/Sampling_Pattern_Design/Mask_16.mat'
Info = scipy.io.whosmat(Mask_16_64_Address)
print(Info)
datam = scipy.io.loadmat(Mask_16_64_Address)
Mask_16_64 = np.asarray(datam ['pattern'])
print(Mask_16_64)

Mask_8_64_Address = '/content/drive/MyDrive/Spike_Restoration/Main_Work_not_Proposal/Sampling_Pattern_Design/Mask_8.mat'
Info = scipy.io.whosmat(Mask_8_64_Address)
print(Info)
datam = scipy.io.loadmat(Mask_8_64_Address)
Mask_8_64 = np.asarray(datam ['pattern'])
print(Mask_8_64)

Mask_4_64_Address = '/content/drive/MyDrive/Spike_Restoration/Main_Work_not_Proposal/Sampling_Pattern_Design/Mask_4.mat'
Info = scipy.io.whosmat(Mask_4_64_Address)
print(Info)
datam = scipy.io.loadmat(Mask_4_64_Address)
Mask_4_64 = np.asarray(datam ['pattern'])
print(Mask_4_64)

Mask_2_64_Address = '/content/drive/MyDrive/Spike_Restoration/Main_Work_not_Proposal/Sampling_Pattern_Design/Mask_2.mat'
Info = scipy.io.whosmat(Mask_2_64_Address)
print(Info)
datam = scipy.io.loadmat(Mask_2_64_Address)
Mask_2_64 = np.asarray(datam ['pattern'])
print(Mask_2_64)

[('pattern', (1, 64), 'double')]
[[0 0 0 0 0 0 0 0 0 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 1
  1 1 1 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 1 0 1 0 0 0 0 1 1 0]]
[('pattern', (1, 64), 'double')]
[[0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 1 1 1 1 0 1 1 1 1 1 0 0 0 0 0 0 0 0 1 1 0
  1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0]]
[('pattern', (1, 64), 'double')]
[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 1 0 1 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0]]
[('pattern', (1, 64), 'double')]
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 0. 0. 0. 1. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
[('pattern', (1, 64), 'double')]
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


Equispaced sampling pattern

In [ ]:
N = 64
Mask_32_64_Reg = np.zeros((1,N),int)
for i in range (0,N):
  if i % 2 == 0:
    Mask_32_64_Reg[0,i] =1
print(Mask_32_64_Reg)

Mask_16_64_Reg = np.zeros((1,N),int)
for i in range (0,N):
  if i % 4 == 0:
    Mask_16_64_Reg[0,i] =1
print(Mask_16_64_Reg)

Mask_8_64_Reg = np.zeros((1,N),int)
for i in range (0,N):
  if i % 8 == 0:
    Mask_8_64_Reg[0,i] =1
print(Mask_8_64_Reg)

[[1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0
  1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0]]
[[1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0
  1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0]]
[[1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0
  0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]]


ZF

In [ ]:
# Mask_16 is just a placeholder name for any of the 8,16,32 VD and reg masks

Mask_16 = Mask_4_64

Train_Data_X_1 = np.zeros(Train_Data_Y_1.shape)
for i in range(1,Train_Data_Y_1.shape[0]):
 Train_Data_X_1[i] = Mask_16 * Train_Data_Y_1[i]

Val_Data_X = np.zeros(Val_Data_Y.shape)
for i in range(1,Val_Data_Y.shape[0]):
 Val_Data_X[i] = Mask_16 * Val_Data_Y[i]

#Unet 19

In [ ]:
def unet(input_size):

    input = Input(input_size)

    # 256 x 256 x 64
    conv1 = Conv1D(64, 3, activation="relu", padding="same")(input)
    conv1 = Conv1D(64, 3, activation="relu", padding="same")(conv1)

    # 128 x 128 x 128
    MaxPool1 = MaxPooling1D(name='block1_pool')(conv1)
    conv2 = Conv1D(128, 3, activation="relu", padding="same")(MaxPool1)
    conv2 = Conv1D(128, 3, activation="relu", padding="same")(conv2)

    # 64 x 64 x 256
    MaxPool2 = MaxPooling1D(name='block2_pool')(conv2)
    conv3 = Conv1D(256, 3, activation="relu", padding="same")(MaxPool2)
    conv3 = Conv1D(256, 3, activation="relu", padding="same")(conv3)

    # 32 x 32 x 512
    MaxPool3 = MaxPooling1D(name='block3_pool')(conv3)
    conv4 = Conv1D(512, 3, activation="relu", padding="same")(MaxPool3)
    conv4 = Conv1D(512, 3, activation="relu", padding="same")(conv4)

    # 16 x 16 x 1024
    MaxPool4 = MaxPooling1D(name='block4_pool')(conv4)
    conv5 = Conv1D(1024, 3, activation="relu", padding="same")(MaxPool4)
    conv5 = Conv1D(1024, 3, activation="relu", padding="same")(conv5)

    # 32 x 32 x 512
    deconv4 = Conv1DTranspose(512, 3, strides=2, padding="same")(conv5)
    uconv4 = concatenate([deconv4, conv4])
    uconv4 = Conv1D(512, 3, activation="relu", padding="same")(uconv4)
    uconv4 = Conv1D(512, 3, activation="relu", padding="same")(uconv4)

    # 64 x 64 x 256
    deconv3 = Conv1DTranspose(256, 3, strides=2, padding="same")(uconv4)
    uconv3 = concatenate([deconv3, conv3])
    uconv3 = Conv1D(256, 3, activation="relu", padding="same")(uconv3)
    uconv3 = Conv1D(256, 3, activation="relu", padding="same")(uconv3)

    # 128 x 128 x 128
    deconv2 = Conv1DTranspose(128, 3, strides=2, padding="same")(uconv3)
    uconv2 = concatenate([deconv2, conv2])
    uconv2 = Conv1D(128, 3, activation="relu", padding="same")(uconv2)
    uconv2 = Conv1D(128, 3, activation="relu", padding="same")(uconv2)

    # 256 x 256 x 64
    deconv1 = Conv1DTranspose(64, 3, strides=2, padding="same")(uconv2)
    uconv1 = concatenate([deconv1, conv1])
    uconv1 = Conv1D(64, 3, activation="relu", padding="same")(uconv1)
    uconv1 = Conv1D(64, 3, activation="relu", padding="same")(uconv1)

    # 256 x 256 x 1
    # uconv1 = keras.layers.BatchNormalization()(uconv1)
    # uconv1 = keras.layers.Dropout(rate = 0.5)(uconv1)

    output_layer = Conv1D(1, 1, padding="same", activation="linear")(uconv1)

    model = Model(input,output_layer)

    return model

#AE

In [ ]:
def ae(input_size,M):
    input = Input(input_size)
    x = input
    N = input_size[0]
    # Encoder
    x = keras.layers.Flatten(input_shape=input_size)(x)
    x = keras.layers.Dense(M, activation='relu')(x)
    # activity_regularizer=keras.regularizers.L2(1e-4))(x)
    #Decoder
    x = keras.layers.Dense(N, activation='linear')(x)
    # activity_regularizer=keras.regularizers.L2(1e-4))(x)
    # x = tf.expand_dims(x, axis=2) #Not working anymore as of Nov 2025
    x = keras.ops.expand_dims(x, axis=2)

    # Create the model
    model = Model(inputs=input, outputs=x)
    return model

#Train AE

In [ ]:
N = 64
M = 2
input_size = (N, 1)
model = ae(input_size,M)
# model.summary()
Optimizer = keras.optimizers.RMSprop(learning_rate=1e-5)
model.compile(optimizer = Optimizer, loss = 'MSE',
              metrics = ['RootMeanSquaredError'])
early_stopping = EarlyStopping(patience=20, monitor = 'val_RootMeanSquaredError',
                               mode = 'min', restore_best_weights=True)
model_checkpoint = ModelCheckpoint(filepath = '/content/drive/MyDrive/Spike_Restoration/Main_Work_not_Proposal/AE2_2.keras',
                                   monitor = 'val_RootMeanSquaredError', mode = 'min', save_best_only=True)
#%%time
# X_train = np.expand_dims(Train_Data_X_1, axis=2)
Y_train = np.expand_dims(Train_Data_Y_1, axis=2)
# X_val = np.expand_dims(Val_Data_X, axis=2)
Y_val = np.expand_dims(Val_Data_Y, axis=2)

history = model.fit((Y_train),(Y_train),
                    batch_size = 32,
                    epochs=500*2,
                    validation_data=(Y_val, Y_val),
                    callbacks=[early_stopping, model_checkpoint])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/1000
4516/4516 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - RootMeanSquaredError: 52.5290 - loss: 2759.4595 - val_RootMeanSquaredError: 91.7268 - val_loss: 8413.8066
Epoch 2/1000
4516/4516 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - RootMeanSquaredError: 51.5990 - loss: 2662.5947 - val_RootMeanSquaredError: 90.1761 - val_loss: 8131.7231
Epoch 3/1000
4516/4516 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - RootMeanSquaredError: 50.4008 - loss: 2540.3508 - val_RootMeanSquaredError: 86.7759 - val_loss: 7530.0483
Epoch 4/1000
4516/4516 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - RootMeanSquaredError: 48.2027 - loss: 2323.6265 - val_RootMeanSquaredError: 81.9954 - val_loss: 6723.2432
Epoch 5/1000
4516/4516 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - RootMeanSquaredError: 45.5943 - loss: 2078.9871 - val_RootMeanSquaredError: 76.9908 - val_loss: 5927.5811
Epoch 6/1000
4516/4516 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - RootMeanSquaredError: 43.3212 - loss: 1876.8035 - val_RootMeanSquaredError: 73.2323 - val_loss: 5362.9688
Epoch 7/1000

#Train ZFU-net

In [ ]:
N = 64
input_size = (N, 1)
model = unet(input_size)
# model.summary()

Optimizer = keras.optimizers.RMSprop(learning_rate=1e-5)
model.compile(optimizer = Optimizer, loss = 'MSE',
              metrics = ['RootMeanSquaredError'])
early_stopping = EarlyStopping(patience=20, monitor = 'val_RootMeanSquaredError',
                             mode = 'min',   restore_best_weights=True)
model_checkpoint = ModelCheckpoint(filepath = '/content/drive/MyDrive/Spike_Restoration/Main_Work_not_Proposal/Unet_4.keras',
                      monitor = 'val_RootMeanSquaredError', mode='min', save_best_only=True)
#%%time
X_train = np.expand_dims(Train_Data_X_1, axis=2)
Y_train = np.expand_dims(Train_Data_Y_1, axis=2)
X_val = np.expand_dims(Val_Data_X, axis=2)
Y_val = np.expand_dims(Val_Data_Y, axis=2)

history = model.fit((X_train),(Y_train),
                    batch_size = 32,
                    epochs=500*2,
                    validation_data=(X_val, Y_val),
                    callbacks=[early_stopping, model_checkpoint])

Epoch 1/1000
4516/4516 ━━━━━━━━━━━━━━━━━━━━ 99s 20ms/step - RootMeanSquaredError: 34.8509 - loss: 1238.1459 - val_RootMeanSquaredError: 55.2588 - val_loss: 3053.5303
Epoch 2/1000
4516/4516 ━━━━━━━━━━━━━━━━━━━━ 82s 18ms/step - RootMeanSquaredError: 28.5064 - loss: 812.6468 - val_RootMeanSquaredError: 52.1239 - val_loss: 2716.9038
Epoch 3/1000
4516/4516 ━━━━━━━━━━━━━━━━━━━━ 82s 18ms/step - RootMeanSquaredError: 27.9017 - loss: 778.5712 - val_RootMeanSquaredError: 52.2027 - val_loss: 2725.1255
Epoch 4/1000
4516/4516 ━━━━━━━━━━━━━━━━━━━━ 82s 18ms/step - RootMeanSquaredError: 27.4729 - loss: 754.7911 - val_RootMeanSquaredError: 53.2609 - val_loss: 2836.7205
Epoch 5/1000
4516/4516 ━━━━━━━━━━━━━━━━━━━━ 82s 18ms/step - RootMeanSquaredError: 27.1958 - loss: 739.6483 - val_RootMeanSquaredError: 51.8141 - val_loss: 2684.7031
Epoch 6/1000
4516/4516 ━━━━━━━━━━━━━━━━━━━━ 82s 18ms/step - RootMeanSquaredError: 26.9768 - loss: 727.7797 - val_RootMeanSquaredError: 52.9758 - val_loss: 2806.4380
Epoch 7/1

#Fine-tune ZFPU-Net
TL took only 6:34 minutes. Report this + backbone training time.

In [ ]:
M = np.sum(Mask_16)
ZFUNet_address = '/content/drive/MyDrive/Spike_Restoration/Main_Work_not_Proposal/Unet_'+ str(int(M)) + '.keras'
model = keras.models.load_model(ZFUNet_address)
for layer in model.layers:
  layer.trainable = True

PersDataAddr = '/content/drive/MyDrive/Spike_Restoration/Main_Work_not_Proposal/Splitted_Data_for_paper1/SNDR/Personalization'
TestRatIDs = os.listdir(PersDataAddr)

for j in TestRatIDs:
  Pers_Address = PersDataAddr + '/' + j + '/Pers_Data.mat'
  Info = scipy.io.whosmat(Pers_Address)
  print(Info)
  datam = scipy.io.loadmat(Pers_Address)
  Train_Data_Y = np.asarray(datam ['Pers_Data'])
  Test_Address = PersDataAddr + '/' + j + '/Test_Data.mat'
  Info = scipy.io.whosmat(Test_Address)
  print(Info)
  datam = scipy.io.loadmat(Test_Address)
  Test_Data_Y = np.asarray(datam ['Test_Data'])
  ####
  # Mask_16 = Mask_32_64
  Train_Data_X = np.zeros(Train_Data_Y.shape)
  for i in range(1,Train_Data_Y.shape[0]):
   Train_Data_X[i] = Mask_16 * Train_Data_Y[i]
  Test_Data_X = np.zeros(Test_Data_Y.shape)
  for i in range(1,Test_Data_Y.shape[0]):
   Test_Data_X[i] = Mask_16 * Test_Data_Y[i]
  ####
  X_train = np.expand_dims(Train_Data_X, axis=2)
  Y_train = np.expand_dims(Train_Data_Y, axis=2)
  X_test = np.expand_dims(Test_Data_X, axis=2)
  Y_test = np.expand_dims(Test_Data_Y, axis=2)
  ###
  Optimizer = keras.optimizers.RMSprop(learning_rate=1e-6)
  model.compile(optimizer = Optimizer, loss = 'MSE',
              metrics = ['RootMeanSquaredError'])
  early_stopping = EarlyStopping(patience=20, monitor = 'val_RootMeanSquaredError',
                                restore_best_weights=True)
  model_checkpoint = ModelCheckpoint(filepath = PersDataAddr + '/' + j + '/ZFPU-Net_' + str(int(M)) + '.keras',)
                                    # monitor = 'val_root_mean_squared_error', save_best_only=True)
  #%%time
  history = model.fit((X_train),(Y_train),
                    batch_size = 32,
                    epochs=20,
                    validation_data=(X_test, Y_test),
                    callbacks=[model_checkpoint])

[('Pers_Data', (239, 64), 'double')]
[('Test_Data', (2172, 64), 'double')]
Epoch 1/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 26s 2s/step - RootMeanSquaredError: 14.7939 - loss: 219.0752 - val_RootMeanSquaredError: 13.3828 - val_loss: 179.0993
Epoch 2/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 156ms/step - RootMeanSquaredError: 14.1898 - loss: 201.5952 - val_RootMeanSquaredError: 13.2591 - val_loss: 175.8036
Epoch 3/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 147ms/step - RootMeanSquaredError: 14.3181 - loss: 205.2781 - val_RootMeanSquaredError: 13.1769 - val_loss: 173.6298
Epoch 4/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 140ms/step - RootMeanSquaredError: 14.0617 - loss: 198.1635 - val_RootMeanSquaredError: 13.1230 - val_loss: 172.2141
Epoch 5/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 148ms/step - RootMeanSquaredError: 13.7068 - loss: 188.0100 - val_RootMeanSquaredError: 13.0904 - val_loss: 171.3595
Epoch 6/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 148ms/step - RootMeanSquaredError: 14.2248 - loss: 202.6715 - val_RootMeanSquaredError: 13.0649 - val_loss: 

#Train ZFPU-Net0

In [ ]:
M = np.sum(Mask_16)
N = 64
input_size = (N, 1)
model = unet(input_size)

PersDataAddr = '/content/drive/MyDrive/Spike_Restoration/Main_Work_not_Proposal/Splitted_Data_for_paper1/SNDR/Personalization'
TestRatIDs = os.listdir(PersDataAddr)

for j in TestRatIDs:
  Pers_Address = PersDataAddr + '/' + j + '/Pers_Data.mat'
  Info = scipy.io.whosmat(Pers_Address)
  print(Info)
  datam = scipy.io.loadmat(Pers_Address)
  Train_Data_Y = np.asarray(datam ['Pers_Data'])
  Test_Address = PersDataAddr + '/' + j + '/Test_Data.mat'
  Info = scipy.io.whosmat(Test_Address)
  print(Info)
  datam = scipy.io.loadmat(Test_Address)
  Test_Data_Y = np.asarray(datam ['Test_Data'])
  ####
  # Mask_16 = Mask_32_64
  Train_Data_X = np.zeros(Train_Data_Y.shape)
  for i in range(1,Train_Data_Y.shape[0]):
   Train_Data_X[i] = Mask_16 * Train_Data_Y[i]
  Test_Data_X = np.zeros(Test_Data_Y.shape)
  for i in range(1,Test_Data_Y.shape[0]):
   Test_Data_X[i] = Mask_16 * Test_Data_Y[i]
  ####
  X_train = np.expand_dims(Train_Data_X, axis=2)
  Y_train = np.expand_dims(Train_Data_Y, axis=2)
  X_test = np.expand_dims(Test_Data_X, axis=2)
  Y_test = np.expand_dims(Test_Data_Y, axis=2)
  ###
  Optimizer = keras.optimizers.RMSprop(learning_rate=1e-6)
  model.compile(optimizer = Optimizer, loss = 'MSE',
              metrics = ['RootMeanSquaredError'])
  early_stopping = EarlyStopping(patience=20, monitor = 'val_root_mean_squared_error',
                                restore_best_weights=True)
  model_checkpoint = ModelCheckpoint(filepath = PersDataAddr + '/' + j + '/ZFPU-Net0_' + str(M) + '.keras',)
                                    # monitor = 'val_root_mean_squared_error', save_best_only=True)
  #%%time
  history = model.fit((X_train),(Y_train),
                    batch_size = 32,
                    epochs=20,
                    validation_data=(X_test, Y_test),
                    callbacks=[model_checkpoint])

[('Pers_Data', (239, 64), 'double')]
[('Test_Data', (2172, 64), 'double')]
Epoch 1/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 13s 773ms/step - RootMeanSquaredError: 29.1996 - loss: 852.6484 - val_RootMeanSquaredError: 27.8132 - val_loss: 773.5757
Epoch 2/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 171ms/step - RootMeanSquaredError: 28.9453 - loss: 838.0019 - val_RootMeanSquaredError: 27.7983 - val_loss: 772.7428
Epoch 3/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 190ms/step - RootMeanSquaredError: 29.4187 - loss: 865.6309 - val_RootMeanSquaredError: 27.7848 - val_loss: 771.9958
Epoch 4/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 171ms/step - RootMeanSquaredError: 29.0324 - loss: 842.8973 - val_RootMeanSquaredError: 27.7719 - val_loss: 771.2769
Epoch 5/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 155ms/step - RootMeanSquaredError: 28.9830 - loss: 840.1454 - val_RootMeanSquaredError: 27.7592 - val_loss: 770.5734
Epoch 6/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 161ms/step - RootMeanSquaredError: 28.9013 - loss: 835.3133 - val_RootMeanSquaredError: 27.7464 - val_los

#Validate ZFUnet

In [ ]:
def sndr(x0,x):
  return 20*np.log10(np.linalg.norm(x0)/np.linalg.norm(x-x0))

model = keras.models.load_model("/content/drive/MyDrive/Spike_Restoration/Main_Work_not_Proposal/Unet_8.h5")
X_test = np.expand_dims(Test_Data_X, axis=2)
prediction = model.predict(X_val) #Y for AE, X for U-Net

SNDR = np.zeros([Test_Data_Y.shape[0]])
# There's much difference between inputting matrices and tensors. Be careful.
for i in range (0,Test_Data_X.shape[0]):
  SNDR[i] = sndr(Test_Data_Y[i], prediction[i,:,0])

print(np.mean(SNDR), np.std(SNDR))

TypeError: Error when deserializing class 'Conv1DTranspose' using config={'name': 'conv1d_transpose', 'trainable': True, 'dtype': 'float32', 'filters': 512, 'kernel_size': [3], 'strides': [2], 'padding': 'same', 'data_format': 'channels_last', 'dilation_rate': [1], 'groups': 1, 'activation': 'linear', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'kernel_regularizer': None, 'bias_regularizer': None, 'activity_regularizer': None, 'kernel_constraint': None, 'bias_constraint': None, 'output_padding': None}.

Exception encountered: Unrecognized keyword arguments passed to Conv1DTranspose: {'groups': 1}

#Validate AE

In [ ]:
def sndr(x0,x):
  return 20*np.log10(np.linalg.norm(x0)/np.linalg.norm(x-x0))

model = keras.models.load_model("/content/drive/MyDrive/Spike_Restoration/Main_Work_not_Proposal/AE_16.h5")
Y_test = np.expand_dims(Test_Data_Y, axis=2)
prediction = model.predict(Y_val) #Y for AE, X for U-Net

SNDR = np.zeros([Test_Data_Y.shape[0]])
# There's much difference between inputting matrices and tensors. Be careful.
for i in range (0,Test_Data_X.shape[0]):
  SNDR[i] = sndr(Test_Data_Y[i], prediction[i,:,0])

print(np.mean(SNDR), np.std(SNDR))

/usr/local/lib/python3.10/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


ValueError: Unknown layer: 'TFOpLambda'. Please ensure you are using a `keras.utils.custom_object_scope` and that this object is included in the scope. See https://www.tensorflow.org/guide/keras/save_and_serialize#registering_the_custom_object for details.

In [ ]:
str(int(M))

'2'